# IGDB 002: IGDB field review

**Per-column EDA on pipeline IGDB artifacts**

Profiles IGDB game columns in `lookups/games.parquet` (Job 1) and Job 2 enriched taxonomy columns (`*_names`, `*_names__use`, `*_names__use_pooled`). Field docs: [`field_docs.py`](../../src/steam_review_ml/igdb/field_docs.py).


# Executive Summary

**Question:**  
For each IGDB game column in our joined parquet, what is the **value shape**, **typical content**, and **likely v2 utility**?

**Result:**  
315 catalog games. Job 1 `lookups/games.parquet` has 17 IGDB fields plus `summary__use` / `storyline__use` (512-d USE, 100% coverage). V2-core FK fields are well populated: genres 100%, themes 98.4%, keywords 91.1%, game_modes 100%, player_perspectives 95.9%. Franchises are sparse (30%) — not a V2a candidate. Job 2 enriched adds 15 taxonomy columns (`*_names`, `*_names__use`, `*_names__use_pooled`); pooled vectors are 512-d L2-normalized and mirror FK coverage per field. Manual mock for Wallpaper Engine (`431960`) resolves (e.g. genres → Indie, Simulator).

**Recommendation / Decision:**  
**Proceed with V2a.** Wire metadata overlap on FK id sets (`genres`, `themes`, `keywords`, `game_modes`, `player_perspectives`) for category-bridge retrieval. Optionally ablate `*_names__use_pooled` cosine vs Jaccard later. Use `summary__use` for V2b text similarity. Defer `franchises` and non-core id-list fields.


# Data Sources

**Job 1:** `artifacts/igdb/lookups/games.parquet` (`recs_job_igdb_games.py`)

**Job 2:** `artifacts/igdb/igdb_games__enriched.parquet` (`recs_job_igdb_games_enriched.py`) — taxonomy `*_names`, `*_names__use`, `*_names__use_pooled`

**Join context:** `artifacts/igdb/igdb_join_report.json` (optional)

**Pipeline refresh:**
```bash
python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json
python scripts/recs_job_igdb_games_enriched.py configs/recs_job_igdb_games_enriched.json
```


# Notebook Roadmap

1. Setup + load Job 1 / Job 2 parquets
2. Overview table (Job 1 IGDB fields)
3. Per-field profiles (Job 1)
4. Job 2 enriched taxonomy columns (`*_names__use_pooled`, etc.)
5. Key Findings (manual)


# Analysis


## Setup


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.igdb.constants import (
    IGDB_GAMES_ENRICHED_PARQUET,
    IGDB_GAMES_PARQUET,
    STEAM_JOIN_COLS,
    TAXONOMY_RESOLVE_FIELDS,
    V2_CORE_FIELDS,
)
from steam_review_ml.igdb.entity_lookup import (
    taxonomy_names_column,
    taxonomy_names_embedding_column,
    taxonomy_names_embedding_pooled_column,
)
from steam_review_ml.igdb.field_docs import GAME_FIELD_DOCS, V2_FIELD_NOTES

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
IGDB_DIR = REPO_ROOT / "artifacts/igdb"
PARQUET_PATH = IGDB_DIR / IGDB_GAMES_PARQUET
ENRICHED_PATH = IGDB_DIR / IGDB_GAMES_ENRICHED_PARQUET
REPORT_PATH = IGDB_DIR / "igdb_join_report.json"

SAMPLE_APP_IDS = [753420, 646910, 512900, 431960]

print(f"REPO_ROOT={REPO_ROOT}")


REPO_ROOT=/home/ryanr/workspace/steam_recommendations


In [2]:
if not PARQUET_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {PARQUET_PATH}. Run:\n"
        "  python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json"
    )

df = pd.read_parquet(PARQUET_PATH)
join_report = json.loads(REPORT_PATH.read_text(encoding="utf-8")) if REPORT_PATH.is_file() else {}

enriched_df: pd.DataFrame | None = None
ENRICHED_TAXONOMY_FIELDS: list[str] = []
if ENRICHED_PATH.is_file():
    enriched_df = pd.read_parquet(ENRICHED_PATH)
    ENRICHED_TAXONOMY_FIELDS = [
        col
        for field in TAXONOMY_RESOLVE_FIELDS
        for col in (
            taxonomy_names_column(field),
            taxonomy_names_embedding_column(field),
            taxonomy_names_embedding_pooled_column(field),
        )
        if col in enriched_df.columns
    ]
else:
    print(
        f"Skip enriched profiles: missing {ENRICHED_PATH}\n"
        "  python scripts/recs_job_igdb_games_enriched.py configs/recs_job_igdb_games_enriched.json"
    )

IGDB_GAME_FIELDS = sorted(c for c in df.columns if c not in STEAM_JOIN_COLS and c != "igdb_name")
absent_from_parquet = sorted(set(GAME_FIELD_DOCS) - set(IGDB_GAME_FIELDS) - {"name"})

print(f"rows={len(df)} cols={len(df.columns)}")
print(f"IGDB_GAME_FIELDS ({len(IGDB_GAME_FIELDS)}): {IGDB_GAME_FIELDS}")
if enriched_df is not None:
    print(f"enriched rows={len(enriched_df)} enriched_taxonomy_fields={ENRICHED_TAXONOMY_FIELDS}")
print(f"catalog match rate: {join_report.get('match_rate', 'n/a')}")
if absent_from_parquet:
    print(f"Fields in GAME_FIELD_DOCS but not in parquet: {absent_from_parquet}")


rows=315 cols=22
IGDB_GAME_FIELDS (17): ['age_ratings', 'collections', 'franchises', 'game_engines', 'game_modes', 'game_type', 'genres', 'involved_companies', 'keywords', 'multiplayer_modes', 'player_perspectives', 'storyline', 'storyline__use', 'summary', 'summary__use', 'tags', 'themes']
enriched rows=315 enriched_taxonomy_fields=['genres_names', 'genres_names__use', 'genres_names__use_pooled', 'themes_names', 'themes_names__use', 'themes_names__use_pooled', 'keywords_names', 'keywords_names__use', 'keywords_names__use_pooled', 'game_modes_names', 'game_modes_names__use', 'game_modes_names__use_pooled', 'player_perspectives_names', 'player_perspectives_names__use', 'player_perspectives_names__use_pooled']
catalog match rate: 1.0
Fields in GAME_FIELD_DOCS but not in parquet: ['aggregated_rating', 'aggregated_rating_count', 'alternative_names', 'artworks', 'bundles', 'category', 'checksum', 'collection', 'cover', 'created_at', 'dlcs', 'expanded_games', 'expansions', 'external_games', 

In [3]:
def igdb_field_doc(field: str) -> dict[str, str]:
    doc = GAME_FIELD_DOCS.get(field, {})
    return {
        "igdb_type": doc.get("type", ""),
        "igdb_description": doc.get("description", ""),
        "deprecated": doc.get("deprecated", False),
        "v2_note": V2_FIELD_NOTES.get(field, ""),
    }


def field_populated(value: Any) -> bool:
    if value is None:
        return False
    try:
        if pd.isna(value):
            return False
    except (TypeError, ValueError):
        pass
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, (list, dict, set, tuple, np.ndarray)):
        return len(value) > 0
    return True


def _as_list(value: Any) -> list[Any]:
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple, set)):
        return list(value)
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass
    return [value]


def _is_embedding(value: Any) -> bool:
    if not isinstance(value, np.ndarray):
        return False
    return value.ndim == 1 and np.issubdtype(value.dtype, np.floating)


def value_kind(value: Any) -> str:
    if not field_populated(value):
        return "empty"
    if isinstance(value, str):
        return "string"
    if isinstance(value, (int, np.integer)):
        return "integer"
    if isinstance(value, (float, np.floating)):
        return "float"
    if _is_embedding(value):
        return "embedding"
    if isinstance(value, np.ndarray):
        if value.ndim == 0:
            return value_kind(value.item())
        return "id_list" if np.issubdtype(value.dtype, np.integer) else "array"
    if isinstance(value, list):
        if value and _is_embedding(value[0]):
            return "embedding_list"
        return "id_list" if value and all(isinstance(x, (int, np.integer)) for x in value) else "list"
    return type(value).__name__


def profile_column(series: pd.Series) -> dict[str, Any]:
    populated = series.map(field_populated)
    populated_s = series.loc[populated]
    kinds = populated_s.map(value_kind)
    kind_mode = kinds.mode().iloc[0] if len(kinds) else "empty"

    doc = igdb_field_doc(str(series.name))
    out: dict[str, Any] = {
        "field": series.name,
        "dtype": str(series.dtype),
        "coverage_pct": round(float(populated.mean()) * 100, 2),
        "value_kind": kind_mode,
        "v2_core": series.name in V2_CORE_FIELDS or str(series.name).endswith("__use_pooled"),
        "igdb_type": doc["igdb_type"],
        "igdb_description": doc["igdb_description"],
        "v2_note": doc["v2_note"],
    }

    if kind_mode == "string":
        lens = populated_s.astype(str).str.len()
        out["len_median"] = int(lens.median())
        out["len_p95"] = int(lens.quantile(0.95))
    elif kind_mode in {"integer", "float"}:
        nums = pd.to_numeric(populated_s, errors="coerce").dropna()
        if len(nums):
            out["min"] = float(nums.min())
            out["median"] = float(nums.median())
            out["max"] = float(nums.max())
    elif kind_mode == "id_list":
        lengths = populated_s.map(lambda v: len(_as_list(v)))
        out["list_len_median"] = float(lengths.median())
        out["list_len_p95"] = float(lengths.quantile(0.95))
        out["unique_ids"] = int(pd.Series([x for v in populated_s for x in _as_list(v)]).nunique())
    elif kind_mode == "list":
        lengths = populated_s.map(lambda v: len(_as_list(v)))
        out["list_len_median"] = float(lengths.median())
        out["list_len_p95"] = float(lengths.quantile(0.95))
    elif kind_mode == "embedding":
        dims = populated_s.map(lambda v: int(np.asarray(v).shape[0]))
        out["embed_dim"] = int(dims.mode().iloc[0]) if len(dims) else None
    elif kind_mode == "embedding_list":
        lengths = populated_s.map(lambda v: len(v))
        out["list_len_median"] = float(lengths.median())
        out["list_len_p95"] = float(lengths.quantile(0.95))
        dims = populated_s.map(lambda v: int(np.asarray(v[0]).shape[0]) if len(v) else 0)
        out["embed_dim"] = int(dims.mode().iloc[0]) if len(dims) else None

    return out


def format_sample(value: Any, *, max_len: int = 240) -> str:
    if not field_populated(value):
        return "<empty>"
    if isinstance(value, str):
        text = value.replace("\n", " ")
        return text if len(text) <= max_len else text[: max_len - 3] + "..."
    if _is_embedding(value):
        arr = np.asarray(value, dtype=np.float32)
        norm = float(np.linalg.norm(arr))
        return f"ndarray(shape={arr.shape[0]}, norm={norm:.3f})"
    if isinstance(value, list) and value and _is_embedding(value[0]):
        dim = int(np.asarray(value[0]).shape[0])
        return f"list[{len(value)}]×ndarray({dim},)"
    if isinstance(value, np.ndarray):
        return f"array({value.tolist()})"
    return repr(value)[:max_len]


def display_field_profile(data: pd.DataFrame, field: str) -> None:
    series = data[field]
    prof = profile_column(series)
    doc = igdb_field_doc(field)
    v2_tag = " **V2_CORE**" if prof.get("v2_core") else ""
    dep_tag = " *(deprecated)*" if doc.get("deprecated") else ""
    display(Markdown(f"### `{field}`{v2_tag}{dep_tag}"))
    if doc.get("igdb_type"):
        display(Markdown(f"**IGDB type:** `{doc['igdb_type']}`"))
    if doc.get("igdb_description"):
        display(Markdown(f"**IGDB description:** {doc['igdb_description']}"))
    if str(field).endswith("__use_pooled"):
        display(Markdown("**Job 2 note:** Mean-pooled, L2-normalized USE vector over resolved taxonomy entity names."))
    elif doc.get("v2_note"):
        display(Markdown(f"**V2 note:** {doc['v2_note']}"))

    summary_rows = {
        k: v
        for k, v in prof.items()
        if k not in {"field", "igdb_type", "igdb_description", "v2_note", "v2_core"}
    }
    display(pd.DataFrame([summary_rows]))

    sample_df = (
        data.loc[data["app_id"].isin(SAMPLE_APP_IDS), ["app_id", "app_name", field]]
        .sort_values("app_id")
        .assign(sample=lambda d: d[field].map(format_sample))
        .drop(columns=[field])
    )
    display(sample_df)

    extra = data.loc[series.map(field_populated), field].head(3)
    print("Additional samples:")
    for app_id, val in zip(data.loc[extra.index, "app_id"], extra):
        print(f"  app_id={app_id}: {format_sample(val)}")


## Overview


In [4]:
overview = pd.DataFrame([profile_column(df[c]) for c in IGDB_GAME_FIELDS])
overview = overview.sort_values(["v2_core", "coverage_pct"], ascending=[False, False]).reset_index(drop=True)
display(overview)

display(overview.loc[overview["v2_core"], ["field", "coverage_pct", "value_kind", "igdb_type", "v2_note"]])


,field,dtype,coverage_pct,value_kind,v2_core,igdb_type,igdb_description,v2_note,list_len_median,list_len_p95,unique_ids,min,median,max,len_median,len_p95,embed_dim
0,game_modes,object,100.00,id_list,True,Array of Game Mode IDs,Modes of gameplay,V2a — Jaccard on game mode ID sets.,2.0,4.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,genres,object,100.00,id_list,True,Array of Genre IDs,Genres of the game,V2a — Jaccard on genre ID sets (entity lookup ...,3.0,5.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN
2,summary,str,100.00,string,True,String,A description of the game,"V2b — USE dot(query_review, summary). Ready wi...",NaN,NaN,NaN,NaN,NaN,NaN,357.0,903.0,NaN
3,themes,object,98.41,id_list,True,Array of Theme IDs,Themes of the game,V2a — Jaccard on theme ID sets.,2.0,5.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN
4,player_perspectives,object,95.87,id_list,True,Array of Player Perspective IDs,The main perspective of the player,V2a — Jaccard on perspective ID sets.,1.0,2.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN
5,keywords,object,91.11,id_list,True,Array of Keyword IDs,Associated keywords,V2a — Jaccard on keyword ID sets; ~91% coverag...,12.0,84.0,1707.0,NaN,NaN,NaN,NaN,NaN,NaN
6,franchises,object,29.84,id_list,True,Array of Franchise IDs,Other franchises the game belongs to,V2a — Jaccard on franchise ID sets; sparse (~3...,1.0,2.0,91.0,NaN,NaN,NaN,NaN,NaN,NaN
7,game_type,int64,100.00,integer,False,Reference ID for Game Type,The type of game,Filter DLC/expansion/bundle noise before ranke...,NaN,NaN,NaN,0.0,0.0,11.0,NaN,NaN,NaN
8,storyline__use,object,100.00,embedding,False,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,512.0
9,summary__use,object,100.00,embedding,False,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,512.0


,field,coverage_pct,value_kind,igdb_type,v2_note
0,game_modes,100.00,id_list,Array of Game Mode IDs,V2a — Jaccard on game mode ID sets.
1,genres,100.00,id_list,Array of Genre IDs,V2a — Jaccard on genre ID sets (entity lookup ...
2,summary,100.00,string,String,"V2b — USE dot(query_review, summary). Ready wi..."
3,themes,98.41,id_list,Array of Theme IDs,V2a — Jaccard on theme ID sets.
4,player_perspectives,95.87,id_list,Array of Player Perspective IDs,V2a — Jaccard on perspective ID sets.
5,keywords,91.11,id_list,Array of Keyword IDs,V2a — Jaccard on keyword ID sets; ~91% coverag...
6,franchises,29.84,id_list,Array of Franchise IDs,V2a — Jaccard on franchise ID sets; sparse (~3...


## Per-field review

One subsection per IGDB game column.


In [5]:
for field in IGDB_GAME_FIELDS:
    display_field_profile(df, field)
    print()


### `age_ratings`

**IGDB type:** `Array of Age Rating IDs`

**IGDB description:** The PEGI rating

,dtype,coverage_pct,value_kind,list_len_median,list_len_p95,unique_ids
0,object,85.08,id_list,5.0,7.0,1328


,app_id,app_name,sample
314,431960,Wallpaper Engine,<empty>
2,512900,Streets of Rogue,"array([125538, 195813, 39086, 181520, 57827, 1..."
1,646910,The Crew 2,"array([111035, 111034, 91834, 92244, 215725, 2..."
0,753420,Dungreed,"array([119531, 55417, 98238, 187890])"


...

**IGDB description:** Themes of the game

**V2 note:** V2a — Jaccard on theme ID sets.

,dtype,coverage_pct,value_kind,list_len_median,list_len_p95,unique_ids
0,object,98.41,id_list,2.0,5.0,22


,app_id,app_name,sample
314,431960,Wallpaper Engine,array([38])
2,512900,Streets of Rogue,"array([1, 23, 27, 33])"
1,646910,The Crew 2,"array([1, 38])"
0,753420,Dungreed,array([1])


Additional samples:
  app_id=753420: array([1])
  app_id=646910: array([1, 38])
  app_id=512900: array([1, 23, 27, 33])



## Job 2 enriched taxonomy columns

Resolved from `igdb_games__enriched.parquet`: `{field}_names`, `{field}_names__use` (per-entity vectors), `{field}_names__use_pooled` (mean-pooled field vector).

In [6]:
if enriched_df is None or not ENRICHED_TAXONOMY_FIELDS:
    print("No enriched taxonomy columns to profile.")
else:
    enriched_overview = pd.DataFrame(
        [profile_column(enriched_df[c]) for c in ENRICHED_TAXONOMY_FIELDS]
    )
    enriched_overview = enriched_overview.sort_values(
        ["field"], ascending=True
    ).reset_index(drop=True)
    display(enriched_overview)

    pooled_cols = [c for c in ENRICHED_TAXONOMY_FIELDS if c.endswith("__use_pooled")]
    if pooled_cols:
        display(
            enriched_overview.loc[
                enriched_overview["field"].isin(pooled_cols),
                ["field", "coverage_pct", "value_kind", "embed_dim"],
            ]
        )

    for field in ENRICHED_TAXONOMY_FIELDS:
        display_field_profile(enriched_df, field)
        print()

,field,dtype,coverage_pct,value_kind,v2_core,igdb_type,igdb_description,v2_note,embed_dim
0,game_modes_names,object,100.00,array,False,,,,NaN
1,game_modes_names__use,object,100.00,array,False,,,,NaN
2,game_modes_names__use_pooled,object,100.00,embedding,True,,,,512.0
3,genres_names,object,100.00,array,False,,,,NaN
4,genres_names__use,object,100.00,array,False,,,,NaN
5,genres_names__use_pooled,object,100.00,embedding,True,,,,512.0
6,keywords_names,object,91.11,array,False,,,,NaN
7,keywords_names__use,object,91.11,array,False,,,,NaN
8,keywords_names__use_pooled,object,91.11,embedding,True,,,,512.0
9,player_perspectives_names,object,95.87,array,False,,,,NaN


,field,coverage_pct,value_kind,embed_dim
2,game_modes_names__use_pooled,100.00,embedding,512.0
5,genres_names__use_pooled,100.00,embedding,512.0
8,keywords_names__use_pooled,91.11,embedding,512.0
11,player_perspectives_names__use_pooled,95.87,embedding,512.0
14,themes_names__use_pooled,98.41,embedding,512.0


### `genres_names`

,dtype,coverage_pct,value_kind
0,object,100.0,array


,app_id,app_name,sample
314,431960,Wallpaper Engine,"array(['Indie', 'Simulator'])"
2,512900,Streets of Rogue,"array(['Shooter', 'Role-playing (RPG)', ""Hack ..."
1,646910,The Crew 2,array(['Racing'])
0,753420,Dungreed,"array(['Platform', 'Adventure', 'Indie'])"


...

### `player_perspectives_names__use_pooled` **V2_CORE**

**Job 2 note:** Mean-pooled, L2-normalized USE vector over resolved taxonomy entity names.

,dtype,coverage_pct,value_kind,embed_dim
0,object,95.87,embedding,512


,app_id,app_name,sample
314,431960,Wallpaper Engine,<empty>
2,512900,Streets of Rogue,"ndarray(shape=512, norm=1.000)"
1,646910,The Crew 2,"ndarray(shape=512, norm=1.000)"
0,753420,Dungreed,"ndarray(shape=512, norm=1.000)"


Additional samples:
  app_id=753420: ndarray(shape=512, norm=1.000)
  app_id=646910: ndarray(shape=512, norm=1.000)
  app_id=512900: ndarray(shape=512, norm=1.000)



# Key Findings

**Job 1 — `lookups/games.parquet` (315 rows)**

| V2 field | Coverage | Shape | V2 role |
|----------|----------|-------|---------|
| `genres` | 100% | id_list (median 3 ids) | V2a Jaccard |
| `themes` | 98.4% | id_list | V2a Jaccard |
| `keywords` | 91.1% | id_list | V2a Jaccard |
| `game_modes` | 100% | id_list | V2a Jaccard |
| `player_perspectives` | 95.9% | id_list | V2a Jaccard |
| `summary` / `summary__use` | 100% | text / 512-d vec | V2b text sim |
| `storyline` / `storyline__use` | 100% | text / 512-d vec | V2b (secondary) |
| `franchises` | 29.8% | id_list | Defer |

**Job 2 — `igdb_games__enriched.parquet`**

- All five taxonomy fields have `{field}_names`, `{field}_names__use` (per-entity vector lists), and `{field}_names__use_pooled` (single 512-d mean-pooled vector).
- Pooled-column coverage matches FK coverage per field (e.g. `keywords_names__use_pooled` 91.1%).
- Wallpaper Engine mock (`431960`, `manual_mock`) participates in enriched output with resolved `genres_names` and `themes_names`.

**Decision:** Validations passed — **good to go on V2a** using FK id Jaccard on the five taxonomy fields above.